In [2]:
import numpy as np
import torch
import torch.nn.functional as F
import whisper

In [4]:
MEL_PATH = "/home/lmh/project/whisper/Summer_Bootcamp/minhyeok/whisper_preprocessed/features/000_A0051_S0001_0_G0101_chunk_00000.npy"

mel_np = np.load(MEL_PATH, allow_pickle=False)
mel = torch.from_numpy(mel_np).float()


print("Mel shape:", mel.shape)

model = whisper.load_model("small")
model.eval()

device = model.device

print("모델 n_mels: ", model.dims.n_mels)
print("모델 hidden_size: ", model.dims.n_audio_state)


mel = mel.unsqueeze(0)

mel = mel.to(
    device=device,
    dtype=model.encoder.conv1.weight.dtype,
)

with torch.no_grad():
    conv1_output = model.encoder.conv1(mel)
    conv1_gelu = F.gelu(conv1_output)

print("Conv1 출력:", conv1_output.shape)
print("Conv1 + GELU:", conv1_gelu.shape)


with torch.no_grad():
    conv2_output = model.encoder.conv2(conv1_gelu)
    conv2_gelu = F.gelu(conv2_output)



print("Conv2 출력:", conv2_output.shape)
print("Conv2 + GELU:", conv2_gelu.shape)


encoder_input = conv2_gelu.permute(0, 2, 1)

print("Transformer 입력:", encoder_input.shape)

Mel shape: torch.Size([80, 3000])
모델 n_mels:  80
모델 hidden_size:  768
Conv1 출력: torch.Size([1, 768, 3000])
Conv1 + GELU: torch.Size([1, 768, 3000])
Conv2 출력: torch.Size([1, 768, 1500])
Conv2 + GELU: torch.Size([1, 768, 1500])
Transformer 입력: torch.Size([1, 1500, 768])
